In [8]:
import pandas as pd
import numpy as np

w = pd.read_parquet("../data/interim/balance_sheet_wide.parquet")
w = w[w["Assets"] >= 1e6].copy()                      # décision de l'EDA 2
w = w.sort_values(["cik", "end"]).reset_index(drop=True)
print(w.shape, w["cik"].nunique())
w = w[w["Assets"] >= 1e6].copy()                      # décision de l'EDA 2
w = w[w["cik"] != "0000000000"].copy()                # CIK invalide (champ vide dans 1 JSON) : 5 bilans

(92836, 29) 12219


In [3]:
NOMS = {
    "Assets": "A", "Liabilities": "L", "StockholdersEquity": "E",
    "CashAndCashEquivalentsAtCarryingValue": "C",
    "AssetsCurrent": "AC", "LiabilitiesCurrent": "LC",
    "RetainedEarningsAccumulatedDeficit": "RE",
    "PropertyPlantAndEquipmentNet": "PPE",
    "Goodwill": "GW", "IntangibleAssetsNetExcludingGoodwill": "INT",
}
POSITIFS = ["A", "L", "AC", "LC"]   # toujours > 0  -> variation en log
SIGNES   = ["C", "E"]               # peuvent être 0 ou < 0 -> variation / actifs

def log_var(nouveau, ancien):
    """log(nouveau) - log(ancien), NaN si une valeur est <= 0."""
    return np.log(nouveau.where(nouveau > 0)) - np.log(ancien.where(ancien > 0))

def ratio(num, den):
    """num / den, NaN si le dénominateur est <= 0 (évite les inf)."""
    return num / den.where(den > 0)

def features_et_cibles(w):
    b = w[["cik", "end"] + list(NOMS)].rename(columns=NOMS)
    b = b.sort_values(["cik", "end"]).reset_index(drop=True)
    cols = list(NOMS.values())

    # --- Voisins dans le temps (même entreprise) ---
    g = b.groupby("cik")
    suiv = g[["end"] + cols].shift(-1)     # année t+1  -> UNIQUEMENT pour les cibles
    prec = g[["end"] + cols].shift(1)      # année t-1  -> pour les features
    ok_suiv = (suiv["end"] - b["end"]).dt.days.between(330, 400)
    ok_prec = (b["end"] - prec["end"]).dt.days.between(330, 400)

    out = b[["cik", "end"]].copy()

    # --- FEATURES : uniquement t et t-1 ---
    out["taille"]   = np.log10(b["A"])                    # taille de l'entreprise
    out["L_A"]      = ratio(b["L"], b["A"])               # endettement
    out["E_A"]      = ratio(b["E"], b["A"])               # autonomie financière
    out["C_A"]      = ratio(b["C"], b["A"])               # trésorerie
    out["RE_A"]     = ratio(b["RE"], b["A"])              # bénéfices cumulés
    out["AC_LC"]    = ratio(b["AC"], b["LC"])             # liquidité générale
    out["AC_A"]     = ratio(b["AC"], b["A"])              # part d'actif courant
    out["LC_A"]     = ratio(b["LC"], b["A"])              # part de dettes courantes
    out["PPE_A"]    = ratio(b["PPE"], b["A"])             # intensité en immobilisations
    out["incorp_A"] = ratio(b["GW"].fillna(0) + b["INT"].fillna(0), b["A"])  # goodwill + incorporels
    out["E_negatif"] = (b["E"] < 0).astype(float).where(b["E"].notna())

    for k in POSITIFS:                                    # variations passées (t-1 -> t)
        out[f"d_{k}"] = log_var(b[k], prec[k]).where(ok_prec)
    for k in SIGNES:
        out[f"d_{k}"] = ((b[k] - prec[k]) / prec["A"].where(prec["A"] > 0)).where(ok_prec)

    # --- CIBLES : t -> t+1 ---
    for k in POSITIFS:
        out[f"y_{k}"] = log_var(suiv[k], b[k]).where(ok_suiv)
    for k in SIGNES:
        out[f"y_{k}"] = ((suiv[k] - b[k]) / b["A"]).where(ok_suiv)

    return out

table = features_et_cibles(w)
print(table.shape)
table.head()

(92836, 25)


,cik,end,taille,L_A,E_A,C_A,RE_A,AC_LC,AC_A,LC_A,...,d_AC,d_LC,d_C,d_E,y_A,y_L,y_AC,y_LC,y_C,y_E
0,0000000000,2021-12-31,7.931859,0.099884,0.902748,0.440038,-0.450824,7.835104,0.560875,0.071585,...,NaN,NaN,NaN,NaN,-0.053982,1.057203,-0.431038,1.047769,-0.328467,-0.237988
1,0000000000,2022-12-31,7.908415,0.303444,0.701631,0.117760,-1.121970,1.785694,0.384691,0.215430,...,-0.431038,1.047769,-0.328467,-0.237988,-1.231941,-0.355542,-0.377267,-0.069230,-0.064047,-0.622557
2,0000000000,2023-12-31,7.373390,0.728943,0.271057,0.184119,-6.033565,1.312285,0.904258,0.689071,...,-0.377267,-0.069230,-0.064047,-0.622557,0.104442,0.231908,-0.077896,0.145620,0.070558,-0.080166
3,0000000000,2024-12-31,7.418749,0.828040,0.171960,0.229420,-6.029664,1.049437,0.753536,0.718039,...,-0.077896,0.145620,0.070558,-0.080166,1.602551,1.010952,-0.818651,-0.780545,-0.016052,2.518092
4,0000000000,2025-12-31,8.114728,0.458272,0.541728,0.042968,-2.132975,1.010199,0.066925,0.066249,...,-0.818651,-0.780545,-0.016052,2.518092,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
cibles = [c for c in table if c.startswith("y_")]
print(table[cibles].notna().mean().round(3))          # part des lignes avec une cible
table[cibles].describe(percentiles=[.01, .05, .5, .95, .99]).round(3)

y_A     0.856
y_L     0.836
y_AC    0.655
y_LC    0.652
y_C     0.760
y_E     0.778
dtype: float64


,y_A,y_L,y_AC,y_LC,y_C,y_E
count,79484.000,77572.000,60776.000,60527.000,70570.000,72232.000
mean,0.076,0.113,0.080,0.136,0.115,0.345
std,0.475,0.619,0.738,0.654,3.843,21.987
min,-7.012,-14.035,-11.040,-7.363,-25.288,-45.140
1%,-1.182,-1.510,-1.727,-1.538,-0.531,-0.975
5%,-0.453,-0.510,-0.760,-0.625,-0.218,-0.384
50%,0.036,0.046,0.037,0.077,0.001,0.010
95%,0.774,0.978,1.041,1.076,0.381,0.899
99%,1.740,2.318,2.692,2.472,2.032,4.785
max,9.224,13.630,10.506,9.220,680.858,5599.692


In [5]:
coupure = pd.Timestamp("2018-12-31")
complet = features_et_cibles(w)
tronque = features_et_cibles(w[w["end"] <= coupure])   # on "efface le futur"

features = [c for c in complet if c not in ("cik", "end") and not c.startswith("y_")]

a = complet[complet["end"] <= coupure].set_index(["cik", "end"]).sort_index()
t = tronque.set_index(["cik", "end"]).sort_index()

# 1) Les features doivent être IDENTIQUES
pd.testing.assert_frame_equal(a[features], t[features])
print(f" Les {len(features)} features n'utilisent aucune donnée future")

# 2) Les cibles DOIVENT changer (sinon la cible ne regarderait pas t+1)
diff = (a["y_A"].notna() != t["y_A"].notna()).sum()
print(f"Cibles y_A qui disparaissent quand on efface le futur : {diff:,}  (attendu > 0)")

 Les 17 features n'utilisent aucune donnée future
Cibles y_A qui disparaissent quand on efface le futur : 4,749  (attendu > 0)


In [6]:
ok = table.dropna(subset=["y_A"])
corr = ok[features + cibles].corr(method="spearman").loc[features, cibles]
corr.round(2).style.background_gradient(cmap="RdBu", vmin=-0.5, vmax=0.5)

,y_A,y_L,y_AC,y_LC,y_C,y_E
taille,0.030000,-0.040000,0.000000,-0.070000,-0.010000,0.030000
L_A,0.030000,-0.150000,0.060000,-0.150000,0.050000,0.000000
E_A,-0.020000,0.110000,-0.050000,0.050000,-0.080000,-0.020000
C_A,0.040000,0.040000,-0.060000,0.040000,-0.160000,0.030000
RE_A,0.070000,-0.010000,0.030000,-0.040000,0.010000,0.110000
AC_LC,-0.010000,0.090000,-0.120000,0.160000,-0.110000,-0.010000
AC_A,0.060000,0.050000,-0.050000,-0.000000,-0.050000,0.030000
LC_A,0.080000,-0.090000,0.080000,-0.210000,0.080000,0.050000
PPE_A,-0.010000,-0.040000,0.010000,-0.060000,0.010000,0.030000
incorp_A,0.010000,-0.040000,0.070000,-0.050000,0.030000,0.030000


In [10]:
print(corr.round(2))

            y_A   y_L  y_AC  y_LC   y_C   y_E
taille     0.03 -0.04  0.00 -0.07 -0.01  0.03
L_A        0.03 -0.15  0.06 -0.15  0.05  0.00
E_A       -0.02  0.11 -0.05  0.05 -0.08 -0.02
C_A        0.04  0.04 -0.06  0.04 -0.16  0.03
RE_A       0.07 -0.01  0.03 -0.04  0.01  0.11
AC_LC     -0.01  0.09 -0.12  0.16 -0.11 -0.01
AC_A       0.06  0.05 -0.05 -0.00 -0.05  0.03
LC_A       0.08 -0.09  0.08 -0.21  0.08  0.05
PPE_A     -0.01 -0.04  0.01 -0.06  0.01  0.03
incorp_A   0.01 -0.04  0.07 -0.05  0.03  0.03
E_negatif  0.01 -0.04  0.04  0.03  0.07  0.04
d_A        0.24  0.18  0.08  0.11 -0.01  0.17
d_L        0.13  0.11  0.05  0.05  0.02  0.06
d_AC       0.13  0.08 -0.00  0.06 -0.06  0.11
d_LC       0.06  0.04  0.04 -0.06  0.01  0.07
d_C        0.08  0.06 -0.02  0.06 -0.14  0.08
d_E        0.20  0.14  0.07  0.10 -0.01  0.21


In [7]:
table.to_parquet("../data/processed/model_table.parquet", index=False)
print(table.shape)

(92836, 25)


## Synthèse
- 17 features (ratios à t, taille, variations t-1 → t), 6 cibles (t → t+1).
- Test anti-fuite réussi : features identiques après suppression du futur.
- Corrélations (Spearman) faibles (|ρ| ≤ 0,24) : élan (d_A → y_A = 0,24, d_E → y_E = 0,21)
  et retour à la moyenne (LC_A → y_LC = −0,21, C_A → y_C = −0,16).
- Cibles y_C et y_E très extrêmes (max 680 et 5 600) → winsorizing sur le train.
- 5 bilans au CIK invalide exclus (bug d'ingestion corrigé à la source).